In [4]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 2.5 MB/s eta 0:00:00a 0:00:01


In [5]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
np.random.seed(42)
random.seed(42)

# Parameters
NUM_USERS = 1000
NUM_CARDS = 1200
NUM_TX = 100000

# Helper functions
def random_timestamp(start, end):
    return start + timedelta(seconds=random.randint(0, int((end - start).total_seconds())))

merchant_categories = ["electronics", "gift_cards", "travel", "gaming", "food", "clothing"]

transactions = []

start_date = datetime.now() - timedelta(days=90)
end_date = datetime.now()

for tx_id in range(NUM_TX):
    user_id = random.randint(1, NUM_USERS)
    card_id = random.randint(1, NUM_CARDS)
    timestamp = random_timestamp(start_date, end_date)
    amount = round(np.random.exponential(50), 2)  # most small, few large
    currency = "USD"
    merchant_category = random.choice(merchant_categories)
    merchant_id = random.randint(1, 500)
    merchant_country = random.choice(["US", "CA", "UK", "DE"])
    channel = random.choice(["web", "mobile"])
    device_fingerprint = fake.md5(raw_output=False)
    ip_country = random.choice(["US", "CA", "UK", "DE"])
    avg_amount_30d = round(np.random.uniform(20, 200), 2)
    new_merchant_flag = random.choice([0,1])
    geo_mismatch = 1 if ip_country != merchant_country else 0

    # Simple fraud injection
    fraud_label = 0
    if amount < 5 and new_merchant_flag==1 and random.random()<0.1:
        fraud_label = 1  # card testing
    if geo_mismatch==1 and amount>100 and random.random()<0.05:
        fraud_label = 1  # account takeover
    if merchant_category=="gift_cards" and amount>50 and random.random()<0.07:
        fraud_label = 1  # resale type fraud

    transactions.append([
        tx_id, card_id, user_id, timestamp, amount, currency,
        merchant_id, merchant_category, merchant_country, channel,
        device_fingerprint, ip_country, 0, 0, avg_amount_30d,
        new_merchant_flag, geo_mismatch, fraud_label
    ])

df = pd.DataFrame(transactions, columns=[
    "transaction_id","card_id","user_id","timestamp","amount","currency",
    "merchant_id","merchant_category","merchant_country","channel",
    "device_fingerprint","ip_country","velocity_1h","velocity_24h",
    "avg_amount_30d","new_merchant_flag","geo_mismatch","fraud_label"
])

# Save dataset
df.to_csv("synthetic_credit_card_data.csv", index=False)
print("Dataset generated:", df.shape)

Dataset generated: (100000, 18)
